# Cellpose on Registered Suite2p Binaries

Run cellpose directly on registered suite2p data (data.bin) using `lsp.cellpose()`.
All detected cells are accepted by default (iscell = 1 for all ROIs).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.colors import hsv_to_rgb

import mbo_utilities as mbo
import lbm_suite2p_python as lsp

plt.style.use('dark_background')
plt.rcParams['figure.dpi'] = 120

In [ ]:
# paths - suite2p directory with registered binaries
SUITE2P_PATH = r"\\Rbo-w1\d\W1_DATA\wsnyder\2025-10-16-Females-Shank-Wheel-316902\5\planes\suite2p"
OUTPUT_DIR = Path(r"D:/output/suite2p")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input: {SUITE2P_PATH}")
print(f"Output: {OUTPUT_DIR}")

In [ ]:
# check data shape - reads registered binaries (data.bin) from suite2p planes
arr = mbo.imread(SUITE2P_PATH)
print(f"Shape: {arr.shape}")
print(f"  frames: {arr.shape[0]}")
print(f"  planes: {arr.shape[1] if len(arr.shape) == 4 else 1}")
print(f"  Array type: {type(arr).__name__}")

## Run Cellpose

Parameters matched to cellpose_template.ipynb for comparison.

In [ ]:
# cellpose parameters (matched to cellpose_template.ipynb)
DIAMETER = 4              # expected cell diameter in pixels
CELLPROB_THRESHOLD = -6.0 # lower = more cells detected
FLOW_THRESHOLD = 0.0      # flow error threshold
MIN_SIZE = 2              # min pixels per cell
MAX_SIZE_UM = 35          # max cell diameter in microns

# note: all cells are accepted by default (iscell = 1 for all ROIs)
results = lsp.cellpose(
    input_data=SUITE2P_PATH,
    save_path=OUTPUT_DIR,
    planes=None,  # None = all planes, or [1, 2] for specific planes
    projection="max",
    diameter=DIAMETER,
    cellprob_threshold=CELLPROB_THRESHOLD,
    flow_threshold=FLOW_THRESHOLD,
    min_size=MIN_SIZE,
    max_size_um=MAX_SIZE_UM,
)

print(f"\nTotal cells: {results['n_rois']}")

## Load Results

In [ ]:
# load results for a specific plane (0-indexed)
PLANE = 0
cp = lsp.load_cellpose_results(OUTPUT_DIR, plane_idx=PLANE)
masks = cp["masks"]
proj = cp["projection"]

n_cells = int(masks.max())
print(f"Plane {PLANE}: {n_cells} cells")
print(f"Image: {proj.shape}")

## View Results

In [ ]:
def normalize99(img):
    p1, p99 = np.percentile(img, [1, 99])
    return np.clip((img - p1) / (p99 - p1 + 1e-8), 0, 1)

def mask_overlay(img, masks, alpha=0.4):
    """overlay colored masks on grayscale image"""
    img_norm = normalize99(img)
    rgb = np.stack([img_norm]*3, axis=-1)
    if masks.max() > 0:
        n = masks.max()
        np.random.seed(42)
        colors = np.zeros((n+1, 3))
        for i in range(1, n+1):
            colors[i] = hsv_to_rgb([np.random.rand(), 0.8, 0.9])
        mask_rgb = colors[masks]
        mask_area = masks > 0
        rgb[mask_area] = (1-alpha)*rgb[mask_area] + alpha*mask_rgb[mask_area]
    return np.clip(rgb, 0, 1)

In [ ]:
# full field of view
overlay = mask_overlay(proj, masks)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(normalize99(proj), cmap='gray')
axes[0].set_title('Max Projection (Registered)')
axes[0].axis('off')

axes[1].imshow(overlay)
axes[1].set_title(f'{n_cells} cells detected')
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# zoomed view (100x100 from center)
cy, cx = proj.shape[0] // 2, proj.shape[1] // 2
s = 50  # half-size

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(normalize99(proj[cy-s:cy+s, cx-s:cx+s]), cmap='gray')
axes[0].set_title('Projection (100x100)')
axes[0].axis('off')

axes[1].imshow(overlay[cy-s:cy+s, cx-s:cx+s])
axes[1].set_title('With Masks')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## Cell Size Distribution

In [ ]:
# compute cell diameters from mask areas
if n_cells > 0:
    areas = [np.sum(masks == i) for i in range(1, n_cells + 1)]
    diameters = 2 * np.sqrt(np.array(areas) / np.pi)
    
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(diameters, bins=30, color='cyan', alpha=0.7, edgecolor='white')
    ax.axvline(np.median(diameters), color='yellow', linestyle='--', 
               label=f'Median: {np.median(diameters):.1f} px')
    ax.set_xlabel('Diameter (pixels)')
    ax.set_ylabel('Count')
    ax.set_title(f'Cell Size Distribution (n={n_cells})')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    diameters = np.array([])
    print("No cells detected")

## Summary

In [ ]:
print(f"Input: {SUITE2P_PATH}")
print(f"Output: {OUTPUT_DIR}")
print(f"")
print(f"Parameters:")
print(f"  diameter: {DIAMETER}")
print(f"  cellprob_threshold: {CELLPROB_THRESHOLD}")
print(f"  flow_threshold: {FLOW_THRESHOLD}")
print(f"  min_size: {MIN_SIZE}")
print(f"  max_size_um: {MAX_SIZE_UM}")
print(f"")
print(f"Results:")
print(f"  Total cells (all planes): {results['n_rois']}")
print(f"  Plane {PLANE} cells: {n_cells}")
if len(diameters) > 0:
    print(f"  Median diameter: {np.median(diameters):.1f} px")

## Compare All Planes

In [ ]:
# show cell counts per plane
plane_counts = []
for pr in results['plane_results']:
    plane_counts.append(pr['n_rois'])
    
fig, ax = plt.subplots(figsize=(10, 4))
planes = range(1, len(plane_counts) + 1)
ax.bar(planes, plane_counts, color='lime', alpha=0.7, edgecolor='white')
ax.set_xlabel('Plane')
ax.set_ylabel('Cell Count')
ax.set_title(f'Cells per Plane (Total: {results["n_rois"]})')
ax.set_xticks(planes)
plt.tight_layout()
plt.show()